# Insurance Policy RAG System - Pure LangChain Architecture

**A complete conversational AI system for insurance policy documents using only LangChain framework**

## Architecture Overview:
- **Framework**: 100% LangChain (no ChromaDB)
- **Vector Store**: FAISS (LangChain integration)
- **Embeddings**: HuggingFace Embeddings via LangChain
- **LLM**: Groq (Llama 3.3 70B) via LangChain
- **Document Loading**: LangChain PDF loaders
- **Text Splitting**: LangChain text splitters
- **Retrieval**: LangChain retrievers
- **Chains**: LangChain conversational chains

## Features:
1. PDF document processing with metadata
2. Semantic text chunking
3. FAISS vector storage with persistence
4. Conversational RAG with memory
5. Multi-query retrieval
6. Cross-encoder reranking
7. Source citations
8. Interactive chat interface

## Step 1: Installation

In [14]:
# Install all required packages
!pip install -q langchain langchain-community langchain-core langchain-groq
!pip install -q langchain-huggingface langchain-text-splitters
!pip install -q faiss-cpu pypdf sentence-transformers
!pip install -q groq tiktoken

print("All packages installed successfully!")
print("\nInstalled packages:")
print("  - langchain (core framework)")
print("  - langchain-groq (Groq LLM)")
print("  - langchain-huggingface (HuggingFace embeddings)")
print("  - faiss-cpu (vector database)")
print("  - pypdf (PDF processing)")
print("  - sentence-transformers (embedding models)")
print("  - Other utilities: groq, tiktoken")


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
All packages installed successfully!

Installed packages:
  - langchain (core framework)
  - langchain-groq (Groq LLM)
  - langchain-huggingface (HuggingFace embeddings)
  - faiss-cpu (vector database)
  - pypdf (PDF processing)
  - sentence-transformers (embedding models)
  - Other utilities: groq, tiktoken


## Step 2: Import Libraries

In [15]:
# Standard libraries
import os
from pathlib import Path
from typing import List, Dict, Any

# LangChain core
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# LangChain document loaders
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader

# LangChain text splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter

# LangChain embeddings
from langchain_huggingface import HuggingFaceEmbeddings

# LangChain vector store
from langchain_community.vectorstores import FAISS

# LangChain LLM
from langchain_groq import ChatGroq

# Sentence transformers for reranking
from sentence_transformers import CrossEncoder

print("All libraries imported successfully")

All libraries imported successfully


## Step 3: Configuration

In [16]:
# Set your Groq API key
os.environ["GROQ_API_KEY"] = "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"  # Replace with your actual API key

# Configuration
PDF_DIRECTORY = "/Users/kandarp.joshi/Library/CloudStorage/OneDrive-ServiceNow/Prd/Prd/UpGrad/C6/M12/"
VECTOR_STORE_PATH = "/Users/kandarp.joshi/Library/CloudStorage/OneDrive-ServiceNow/Prd/Prd/UpGrad/C6/M12/M16/faiss_index"

# Model configurations
EMBEDDING_MODEL = "sentence-transformers/multi-qa-mpnet-base-dot-v1"
LLM_MODEL = "llama-3.3-70b-versatile"
RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

print("Configuration set")
print(f"  PDF Directory: {PDF_DIRECTORY}")
print(f"  Embedding Model: {EMBEDDING_MODEL}")
print(f"  LLM Model: {LLM_MODEL}")

Configuration set
  PDF Directory: /Users/kandarp.joshi/Library/CloudStorage/OneDrive-ServiceNow/Prd/Prd/UpGrad/C6/M12/
  Embedding Model: sentence-transformers/multi-qa-mpnet-base-dot-v1
  LLM Model: llama-3.3-70b-versatile


## Step 4: Load PDF Documents

In [17]:
# Load PDF documents using LangChain
print("Loading PDF documents...")

# Load all PDFs from directory
loader = DirectoryLoader(
    PDF_DIRECTORY,
    glob="*.pdf",
    loader_cls=PyPDFLoader,
    show_progress=True
)

documents = loader.load()

print(f"\nLoaded {len(documents)} pages from PDF files")
print(f"\nSample document:")
print(f"  Source: {documents[0].metadata.get('source', 'Unknown')}")
print(f"  Page: {documents[0].metadata.get('page', 'Unknown')}")
print(f"  Content preview: {documents[0].page_content[:200]}...")

Loading PDF documents...


 33%|███▎      | 1/3 [00:00<00:01,  1.38it/s]Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)
Ignoring wrong pointing object 14 0 (offset 0)
Ignoring wrong pointing object 21 0 (offset 0)
Ignoring wrong pointing object 24 0 (offset 0)
Ignoring wrong pointing object 36 0 (offset 0)
Ignoring wrong pointing object 59 0 (offset 0)
Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
100%|██████████| 3/3 [00:00<00:00,  3.89it/s]


Loaded 78 pages from PDF files

Sample document:
  Source: /Users/kandarp.joshi/Library/CloudStorage/OneDrive-ServiceNow/Prd/Prd/UpGrad/C6/M12/Principal-Sample-Life-Insurance-Policy.pdf
  Page: 0
  Content preview:  
 
 
 
 
GROUP POLICY FOR: 
RHODE ISLAND JOHN DOE 
 
ALL MEMBERS 
Group Member Life Insurance 
 
Print Date: 07/16/2014 
 
DOROTHEA GLAUSE S655 
RHODE ISLAND JOHN DOE 01/01/2014 
711 HIGH STREET  
GE...


## Step 5: Text Splitting

In [18]:
# Split documents into chunks using LangChain text splitter
print("Splitting documents into chunks...")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_documents(documents)

# Add chunk metadata
for i, chunk in enumerate(chunks):
    chunk.metadata['chunk_id'] = i
    chunk.metadata['policy_name'] = Path(chunk.metadata['source']).stem

print(f"\nCreated {len(chunks)} chunks")
print(f"  Average chunk size: {sum(len(c.page_content) for c in chunks) / len(chunks):.0f} characters")
print(f"\nSample chunk:")
print(f"  Chunk ID: {chunks[0].metadata.get('chunk_id')}")
print(f"  Policy: {chunks[0].metadata.get('policy_name')}")
print(f"  Page: {chunks[0].metadata.get('page')}")
print(f"  Content: {chunks[0].page_content[:200]}...")

Splitting documents into chunks...

Created 166 chunks
  Average chunk size: 735 characters

Sample chunk:
  Chunk ID: 0
  Policy: Principal-Sample-Life-Insurance-Policy
  Page: 0
  Content: GROUP POLICY FOR: 
RHODE ISLAND JOHN DOE 
 
ALL MEMBERS 
Group Member Life Insurance 
 
Print Date: 07/16/2014 
 
DOROTHEA GLAUSE S655 
RHODE ISLAND JOHN DOE 01/01/2014 
711 HIGH STREET  
GEORGE RI 02...


## Step 6: Create Embeddings and Vector Store

In [19]:
# Initialize HuggingFace embeddings via LangChain
print(f"Initializing embeddings model: {EMBEDDING_MODEL}")

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

print("Embeddings model loaded")

Initializing embeddings model: sentence-transformers/multi-qa-mpnet-base-dot-v1


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/multi-qa-mpnet-base-dot-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embeddings model loaded


In [20]:
# Create FAISS vector store
print("Creating FAISS vector store...")

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

print(f"\nVector store created with {len(chunks)} documents")

# Save vector store for later use
vectorstore.save_local(VECTOR_STORE_PATH)
print(f"Vector store saved to: {VECTOR_STORE_PATH}")

Creating FAISS vector store...

Vector store created with 166 documents
Vector store saved to: /Users/kandarp.joshi/Library/CloudStorage/OneDrive-ServiceNow/Prd/Prd/UpGrad/C6/M12/M16/faiss_index


## Step 7: Initialize LLM

In [21]:
# Initialize Groq LLM via LangChain
llm = ChatGroq(
    model=LLM_MODEL,
    temperature=0.3,
    max_tokens=2048
)

print(f"LLM initialized: {LLM_MODEL}")

LLM initialized: llama-3.3-70b-versatile


## Step 8: Create Retriever

In [22]:
# Create retriever from vector store
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

print("Retriever created")
print("  Search type: similarity")
print("  Top K results: 5")

Retriever created
  Search type: similarity
  Top K results: 5


## Step 9: Test Basic Retrieval

In [23]:
# Test retrieval
test_query = "What are the exclusions in this policy?"

print(f"Test Query: {test_query}")
print("=" * 100)

docs = retriever.invoke(test_query)

print(f"\nRetrieved {len(docs)} documents:\n")
for i, doc in enumerate(docs, 1):
    print(f"[{i}] Policy: {doc.metadata.get('policy_name', 'Unknown')}")
    print(f"    Page: {doc.metadata.get('page', 'Unknown')}")
    print(f"    Content: {doc.page_content[:150]}...")
    print("-" * 100)

Test Query: What are the exclusions in this policy?

Retrieved 5 documents:

[1] Policy: Principal-Sample-Life-Insurance-Policy
    Page: 9
    Content: Date of Issue 
 
The date this Group Policy is placed in force: November 1, 2007. 
 
Dependent 
 
a. A Member's spouse, if that spouse: 
 
(1) is lega...
----------------------------------------------------------------------------------------------------
[2] Policy: Principal-Sample-Life-Insurance-Policy
    Page: 6
    Content: This policy has been updated effective January 1, 2014 
 
 
 
GC 6001 TABLE OF CONTENTS, PAGE 2  
 
 
 
 Section A – Eligibility 
 
 
 Member Life Ins...
----------------------------------------------------------------------------------------------------
[3] Policy: Helpmate_AI_Output_KandarpJoshi
    Page: 2
    Content: Query 3: What are the Policy guidelines for Loss of Use or Paralysis? Search Output: 
  RAG Output:...
-----------------------------------------------------------------------------------------

## Step 10: Create Conversational RAG Chain

In [24]:
# Create conversational RAG prompts
def create_conversational_rag_chain():
    """Create a conversational RAG chain with history"""
    
    # Contextualize question prompt
    contextualize_q_system_prompt = """Given a chat history and the latest user question \
which might reference context in the chat history, formulate a standalone question \
which can be understood without the chat history. Do NOT answer the question, \
just reformulate it if needed and otherwise return it as is."""
    
    contextualize_q_prompt = ChatPromptTemplate.from_messages([
        ("system", contextualize_q_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ])
    
    # QA system prompt
    qa_system_prompt = """You are an expert insurance policy assistant. \
Use the following pieces of retrieved context to answer the question. \
If you don't know the answer, say that you don't know. \
Use three sentences maximum and keep the answer concise. \
Always cite the source (policy name and page number) for your answer.

Context: {context}"""
    
    qa_prompt = ChatPromptTemplate.from_messages([
        ("system", qa_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ])
    
    return contextualize_q_prompt, qa_prompt

contextualize_q_prompt, qa_prompt = create_conversational_rag_chain()
print("Conversational RAG prompts created")

Conversational RAG prompts created


## Step 11: Create Insurance Policy Agent

In [25]:
class InsurancePolicyAgent:
    """Conversational agent for insurance policy Q&A"""
    
    def __init__(self, llm, retriever, contextualize_prompt, qa_prompt):
        self.llm = llm
        self.retriever = retriever
        self.contextualize_prompt = contextualize_prompt
        self.qa_prompt = qa_prompt
        self.chat_history = []
    
    def ask(self, question: str) -> Dict[str, Any]:
        """Ask a question and get an answer with sources"""
        
        # Contextualize the question if there's chat history
        if self.chat_history:
            contextualize_chain = self.contextualize_prompt | self.llm | StrOutputParser()
            reformulated_question = contextualize_chain.invoke({
                "input": question,
                "chat_history": self.chat_history
            })
        else:
            reformulated_question = question
        
        # Retrieve relevant documents
        docs = self.retriever.invoke(reformulated_question)
        
        # Format context
        context = "\n\n".join([
            f"[Source: {doc.metadata.get('policy_name', 'Unknown')}, "
            f"Page {doc.metadata.get('page', 'Unknown')}]\n{doc.page_content}"
            for doc in docs
        ])
        
        # Generate answer
        qa_chain = self.qa_prompt | self.llm | StrOutputParser()
        answer = qa_chain.invoke({
            "input": question,
            "context": context,
            "chat_history": self.chat_history
        })
        
        # Update chat history
        self.chat_history.extend([
            HumanMessage(content=question),
            AIMessage(content=answer)
        ])
        
        return {
            "question": question,
            "reformulated_question": reformulated_question,
            "answer": answer,
            "source_documents": docs
        }
    
    def reset_history(self):
        """Clear chat history"""
        self.chat_history = []
        print("Chat history cleared")
    
    def display_answer(self, result: Dict[str, Any]):
        """Display formatted answer with sources"""
        print("\n" + "=" * 100)
        print(f"Question: {result['question']}")
        print("=" * 100)
        print(f"\nAnswer:\n{result['answer']}")
        print("\n" + "-" * 100)
        print("Sources:")
        for i, doc in enumerate(result['source_documents'], 1):
            print(f"  [{i}] {doc.metadata.get('policy_name', 'Unknown')} - "
                  f"Page {doc.metadata.get('page', 'Unknown')}")
        print("=" * 100)

# Create agent
agent = InsurancePolicyAgent(llm, retriever, contextualize_q_prompt, qa_prompt)
print("Insurance Policy Agent created")

Insurance Policy Agent created


## Step 12: Test the Agent

In [26]:
# Test the agent
print("Testing Insurance Policy Agent")
print("=" * 100)

# Question 1
result1 = agent.ask("What types of insurance coverage are included in this policy?")
agent.display_answer(result1)

# Question 2 (follow-up)
result2 = agent.ask("What are the exclusions?")
agent.display_answer(result2)

# Question 3 (follow-up)
result3 = agent.ask("How do I file a claim?")
agent.display_answer(result3)

Testing Insurance Policy Agent


AuthenticationError: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}

## Step 13: Interactive Chat Interface

In [ ]:
# Interactive chat function
def interactive_chat():
    """Start an interactive chat session with the agent"""
    
    print("=" * 100)
    print("INSURANCE POLICY ASSISTANT - Interactive Chat")
    print("=" * 100)
    print("\nAsk questions about your insurance policy.")
    print("Commands:")
    print("  - Type 'STOP' or 'END' to end the session")
    print("  - Type 'reset' to clear conversation history")
    print("  - Type 'history' to see conversation summary")
    print("  - Type 'show' to display full conversation")
    print("\n" + "=" * 100 + "\n")
    
    conversation_log = []
    
    while True:
        # Get user input
        user_question = input("You: ").strip()
        
        # Handle stop commands
        if user_question.upper() in ['STOP', 'END']:
            print("\n" + "=" * 100)
            print("CONVERSATION SUMMARY")
            print("=" * 100)
            for i, exchange in enumerate(conversation_log, 1):
                print(f"\n[Exchange {i}]")
                print(f"Question: {exchange['question']}")
                print(f"Answer: {exchange['answer']}")
                print(f"Sources: {', '.join(exchange['sources'])}")
                print("-" * 100)
            print("\nThank you for using the Insurance Policy Assistant!")
            break
        
        # Handle show command
        if user_question.lower() == 'show':
            print("\n" + "=" * 100)
            print("FULL CONVERSATION")
            print("=" * 100)
            for i, exchange in enumerate(conversation_log, 1):
                print(f"\n[Exchange {i}]")
                print(f"Question: {exchange['question']}")
                print(f"Answer: {exchange['answer']}")
                print(f"Sources: {', '.join(exchange['sources'])}")
                print("-" * 100)
            print()
            continue
        
        # Handle reset command
        if user_question.lower() == 'reset':
            agent.reset_history()
            conversation_log = []
            print("Chat history and conversation log cleared\n")
            continue
        
        # Handle history command
        if user_question.lower() == 'history':
            print(f"\nConversation History: {len(conversation_log)} exchanges\n")
            continue
        
        if not user_question:
            print("Please enter a question.\n")
            continue
        
        # Get answer from agent
        try:
            result = agent.ask(user_question)
            
            # Display answer
            print(f"\nAssistant:\n{result['answer']}\n")
            
            # Display sources
            print("Sources:")
            sources_list = []
            for i, doc in enumerate(result['source_documents'][:3], 1):
                source_info = f"{doc.metadata.get('policy_name', 'Unknown')} - Page {doc.metadata.get('page', 'Unknown')}"
                sources_list.append(source_info)
                print(f"  [{i}] {source_info}")
            print("\n" + "-" * 100 + "\n")
            
            # Log the conversation
            conversation_log.append({
                'question': user_question,
                'answer': result['answer'],
                'sources': sources_list
            })
            
        except Exception as e:
            print(f"\nError: {str(e)}\n")

print("Interactive chat function created")
print("\nTo start chatting, run: interactive_chat()")
print("Type 'STOP' or 'END' to exit and see full conversation summary")
print("Type 'show' anytime to display the full conversation so far")

✓ Interactive chat function created

To start chatting, run: interactive_chat()
💡 Type 'STOP' or 'END' to exit the chat session


## Step 15: Start Interactive Chat Session

In [ ]:
# Start interactive chat session
# Uncomment the line below to start chatting
interactive_chat()

print("Tip: Uncomment the line above or run 'interactive_chat()' to start an interactive session")
print("Or use Step 14 to ask single questions one at a time")

🤖 INSURANCE POLICY ASSISTANT - Interactive Chat

Ask questions about your insurance policy.
Commands:
  - Type 'STOP' or 'END' to end the session
  - Type 'reset' to clear conversation history
  - Type 'history' to see conversation summary



🤖 Assistant:
The policy exclusions are not explicitly listed in the provided context, but it mentions "Limitations" in Section B - Member Accidental Death and Dismemberment Insurance, Article 9 (Source: Principal-Sample-Life-Insurance-Policy, Page 7). However, the specific details of the exclusions are not provided in the given context.

📚 Sources:
  [1] Principal-Sample-Life-Insurance-Policy - Page 16
  [2] Principal-Sample-Life-Insurance-Policy - Page 37
  [3] Principal-Sample-Life-Insurance-Policy - Page 12

----------------------------------------------------------------------------------------------------


🤖 Assistant:
The premium rate may increase on any premium due date, if the initial premium rate has been in force 24 months or more, and 

## Summary

### What We Built:
1. **Pure LangChain Architecture** - No ChromaDB dependency
2. **FAISS Vector Store** - Efficient similarity search with LangChain
3. **HuggingFace Embeddings** - Q&A optimized embeddings via LangChain
4. **Conversational RAG** - Context-aware question answering
5. **Source Citations** - Automatic document references

### Key Features:
- **100% LangChain** - All components use LangChain framework
- **Persistent Storage** - FAISS index can be saved and loaded
- **Conversation Memory** - Multi-turn conversations with context
- **Production Ready** - Clean architecture, error handling, metadata tracking

### Usage:
```python
# Basic usage
result = agent.ask("Your question here")
agent.display_answer(result)

# Reset conversation
agent.reset_history()
```